# Pump Sensor Data — Initial Exploration

## Dataset
- 220,320 rows of real industrial pump sensor data
- 52 sensor columns (temperatures, pressures, flows, vibrations)
- Source: Kaggle — nphantawee/pump-sensor-data

## Goals
1. Load and inspect data structure
2. Identify data quality issues:
   - Missing values (which columns, what percentage)
   - Duplicates
   - Incorrect datatypes
   - Outliers and unrealistic values

## 0. Imports

In [2]:
import pandas as pd
import numpy as np

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.options.display.max_rows = 60

df = pd.read_csv('../data/sensor.csv')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 220320 entries, 0 to 220319
Data columns (total 55 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Unnamed: 0      220320 non-null  int64  
 1   timestamp       220320 non-null  str    
 2   sensor_00       210112 non-null  float64
 3   sensor_01       219951 non-null  float64
 4   sensor_02       220301 non-null  float64
 5   sensor_03       220301 non-null  float64
 6   sensor_04       220301 non-null  float64
 7   sensor_05       220301 non-null  float64
 8   sensor_06       215522 non-null  float64
 9   sensor_07       214869 non-null  float64
 10  sensor_08       215213 non-null  float64
 11  sensor_09       215725 non-null  float64
 12  sensor_10       220301 non-null  float64
 13  sensor_11       220301 non-null  float64
 14  sensor_12       220301 non-null  float64
 15  sensor_13       220301 non-null  float64
 16  sensor_14       220299 non-null  float64
 17  sensor_15       0 non

## 1. Missing data
Searching for missing data shows that 100% of sensor 15 is missing. Sensors 50 (34.96%), 51 (6.98%), 00 (4.63%), 07 (2.47%), 08 (2.32%), 06 (2.18%) and 09 (2.09%) have also a high percentage of missing data.

In [4]:
df.isna().mean().sort_values(ascending=False) * 100

sensor_15        100.00
sensor_50         34.96
sensor_51          6.98
sensor_00          4.63
sensor_07          2.47
sensor_08          2.32
sensor_06          2.18
sensor_09          2.09
sensor_01          0.17
sensor_30          0.12
sensor_29          0.03
sensor_32          0.03
sensor_18          0.02
sensor_17          0.02
sensor_22          0.02
sensor_25          0.02
sensor_16          0.01
sensor_45          0.01
sensor_39          0.01
sensor_43          0.01
sensor_42          0.01
sensor_40          0.01
sensor_41          0.01
sensor_49          0.01
sensor_47          0.01
sensor_46          0.01
sensor_48          0.01
sensor_44          0.01
sensor_38          0.01
sensor_14          0.01
sensor_26          0.01
sensor_03          0.01
sensor_04          0.01
sensor_11          0.01
sensor_02          0.01
sensor_05          0.01
sensor_10          0.01
sensor_13          0.01
sensor_12          0.01
sensor_21          0.01
sensor_24          0.01
sensor_20       

## 2. Duplicates
In a dataset like this, having equal values within a single column is expected. What is highly unexpected is exact duplicate rows. The only other duplicate check i'm performing is checking for duplicate timestamps.
First block shows that there are no duplicate timestamps.
Second block is more interesting. There are 5745 duplicate rows (counting both duplicate rows). Initial exploration suggested that the duplicates occur on the hour and 1 minute later. Testing of that hypothesis shows that is not the case. More investigating is needed to show if the duplicates are on recurring minutes or that there is a larger delta between duplicates.

In [5]:
ts_duplicates = df['timestamp'].duplicated()
print(f'Duplicate timestamps: {ts_duplicates.sum()}')

Duplicate timestamps: 0


In [6]:
duplicate_rows = df.select_dtypes('float64').duplicated(keep=False)
print(f'Duplicate rows: {duplicate_rows.sum()}')

Duplicate rows: 5745


In [7]:
df.loc[duplicate_rows]

,Unnamed: 0,timestamp,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,sensor_32,sensor_33,sensor_34,sensor_35,sensor_36,sensor_37,sensor_38,sensor_39,sensor_40,sensor_41,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51,machine_status
0,0,2018-04-01 00:00:00,2.47,47.09,53.21,46.31,634.38,76.46,13.41,16.13,15.57,15.05,37.23,47.52,31.12,1.68,419.57,NaN,461.88,466.33,2.57,665.40,398.99,880.00,498.89,975.94,627.67,741.72,848.07,429.04,785.19,684.94,594.44,682.81,680.44,433.70,171.94,341.90,195.07,90.32,40.36,31.51,70.57,30.99,31.77,41.93,39.64,65.68,50.93,38.19,157.99,67.71,243.06,201.39,NORMAL
1,1,2018-04-01 00:01:00,2.47,47.09,53.21,46.31,634.38,76.46,13.41,16.13,15.57,15.05,37.23,47.52,31.12,1.68,419.57,NaN,461.88,466.33,2.57,665.40,398.99,880.00,498.89,975.94,627.67,741.72,848.07,429.04,785.19,684.94,594.44,682.81,680.44,433.70,171.94,341.90,195.07,90.32,40.36,31.51,70.57,30.99,31.77,41.93,39.64,65.68,50.93,38.19,157.99,67.71,243.06,201.39,NORMAL
60,60,2018-04-01 01:00:00,2.45,49.09,53.12,46.01,626.62,73.98,13.50,16.17,15.65,15.12,41.47,53.09,34.38,1.83,419.05,NaN,463.61,469.06,2.59,665.97,399.06,879.61,500.15,981.60,624.41,739.19,848.57,461.32,782.01,702.79,674.07,739.06,680.91,466.23,171.45,333.38,205.25,93.68,48.44,32.55,75.52,31.51,32.29,40.89,36.46,45.43,43.98,39.06,149.59,70.02,229.75,214.12,NORMAL
61,61,2018-04-01 01:01:00,2.45,49.09,53.12,46.01,626.62,73.98,13.50,16.17,15.65,15.12,41.47,53.09,34.38,1.83,419.05,NaN,463.61,469.06,2.59,665.97,399.06,879.61,500.15,981.60,624.41,739.19,848.57,461.32,782.01,702.79,674.07,739.06,680.91,466.23,171.45,333.38,205.25,93.68,48.44,32.55,75.52,31.51,32.29,40.89,36.46,45.43,43.98,39.06,149.59,70.02,229.75,214.12,NORMAL
120,120,2018-04-01 02:00:00,2.44,48.78,52.99,45.83,629.98,73.52,13.69,16.17,15.52,15.08,42.19,55.81,36.60,2.03,417.93,NaN,464.33,463.68,2.56,665.04,398.92,882.51,498.66,979.66,625.24,741.90,850.20,484.18,775.45,696.65,643.06,733.85,681.76,465.64,163.53,319.34,200.93,100.40,46.09,29.17,73.44,33.59,32.29,41.67,43.98,46.59,43.40,37.91,190.10,80.15,228.30,203.70,NORMAL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220141,220141,2018-08-31 21:01:00,2.40,48.61,50.74,43.40,628.74,58.83,15.05,16.70,15.65,15.12,47.44,49.35,42.81,13.19,420.35,NaN,462.31,464.37,2.56,676.40,406.12,893.84,540.78,1107.61,608.43,699.12,798.07,706.27,779.99,496.88,669.91,908.33,876.94,488.62,246.54,526.96,787.07,0.00,41.41,28.91,60.16,29.69,31.25,43.49,44.85,131.94,143.52,46.88,419.27,110.24,NaN,257.52,NORMAL
220200,220200,2018-08-31 22:00:00,2.40,47.96,50.56,43.23,632.29,76.56,15.01,16.65,15.53,15.13,37.58,46.64,36.51,13.12,419.51,NaN,462.00,460.23,2.50,674.90,406.27,892.22,541.39,1108.42,609.39,700.69,796.46,321.12,805.39,490.53,700.00,888.54,915.63,490.80,261.20,543.56,806.65,0.00,46.61,29.95,47.40,29.69,28.91,33.85,47.45,49.19,42.25,36.75,223.96,95.78,NaN,205.44,NORMAL
220201,220201,2018-08-31 22:01:00,2.40,47.96,50.56,43.23,632.29,76.56,15.01,16.65,15.53,15.13,37.58,46.64,36.51,13.12,419.51,NaN,462.00,460.23,2.50,674.90,406.27,892.22,541.39,1108.42,609.39,700.69,796.46,321.12,805.39,490.53,700.00,888.54,915.63,490.80,261.20,543.56,806.65,0.00,46.61,29.95,47.40,29.69,28.91,33.85,47.45,49.19,42.25,36.75,223.96,95.78,NaN,205.44,NORMAL
220260,220260,2018-08-31 23:00:00,2.40,47.79,50.56,42.97,633.45,67.54,15.23,16.70,15.73,15.08,43.03,51.77,37.29,13.37,419.30,NaN,463.76,474.17,2.67,674.07,406.17,891.72,543.62,1107.35,611.35,698.75,795.13,235.49,786.70,489.71,668.52,890.62,900.45,469.60,239.47,511.16,779.77,0.00

In [8]:
duplicate_check = df.loc[duplicate_rows]['timestamp'].str[-5:-3]
duplicate_check.unique()

<StringArray>
['00', '01', '24', '25', '41', '42', '43', '44', '45', '48', '49', '50', '51',
 '52', '53', '54', '55', '13', '14', '12', '57', '58', '16', '17']
Length: 24, dtype: str

## 3. Incorrect datatypes

Earlier df.info() shows two columns with issues. Column 0 'Unnamed' and column 1 'timestamp'.
Column 0 looks to be the indexes from the original dataset. This column will be dropped in later.
Column 1 has datatype 'str'. This column has to be converted to datetime.

## 4. Outliers and unrealistic values

Initial exploration shows several sensors with minimum values of zero. And 28 sensors with maximum values exceeding 3σ from the mean. These warrant further investigation during cleaning.

In [9]:
df.describe()

,Unnamed: 0,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,sensor_32,sensor_33,sensor_34,sensor_35,sensor_36,sensor_37,sensor_38,sensor_39,sensor_40,sensor_41,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51
count,220320.00,210112.00,219951.00,220301.00,220301.00,220301.00,220301.00,215522.00,214869.00,215213.00,215725.00,220301.00,220301.00,220301.00,220301.00,220299.00,0.00,220289.00,220274.00,220274.00,220304.00,220304.00,220304.00,220279.00,220304.00,220304.00,220284.00,220300.00,220304.00,220304.00,220248.00,220059.00,220304.00,220252.00,220304.00,220304.00,220304.00,220304.00,220304.00,220293.00,220293.00,220293.00,220293.00,220293.00,220293.00,220293.00,220293.00,220293.00,220293.00,220293.00,220293.00,143303.00,204937.00
mean,110159.50,2.37,47.59,50.87,43.75,590.67,73.40,13.50,15.84,15.20,14.80,41.47,41.92,29.14,7.08,376.86,NaN,416.47,421.13,2.30,590.83,360.81,796.23,459.79,922.61,556.24,649.14,786.41,501.51,851.69,576.20,614.60,863.32,804.28,486.41,234.97,427.13,593.03,60.79,49.66,36.61,68.84,35.37,35.45,43.88,42.66,43.09,48.02,44.34,150.89,57.12,183.05,202.70
std,63601.05,0.41,3.30,3.67,2.42,144.02,17.30,2.16,2.20,2.04,2.09,12.09,13.06,10.11,6.90,113.21,NaN,126.07,129.16,0.77,199.35,101.97,226.68,154.53,291.84,182.30,220.87,246.66,169.82,313.07,225.76,195.73,283.54,260.60,150.75,88.38,141.77,289.39,37.60,10.54,15.61,21.37,7.90,10.26,11.04,11.58,12.84,15.64,10.44,82.24,19.14,65.26,109.59
min,0.00,0.00,0.00,33.16,31.64,2.80,0.00,0.01,0.00,0.03,0.00,0.00,0.00,0.00,0.00,32.41,NaN,0.00,0.00,0.00,0.00,0.00,95.53,0.00,0.00,0.00,0.00,43.15,0.00,4.32,0.64,0.00,23.96,0.24,6.46,54.88,0.00,2.26,0.00,24.48,19.27,23.44,20.83,22.14,24.48,25.75,26.33,26.33,27.20,26.33,26.62,27.49,27.78
25%,55079.75,2.44,46.31,50.39,42.84,626.62,69.98,13.35,15.91,15.18,15.05,40.71,38.86,28.69,1.54,418.10,NaN,459.45,454.14,2.45,662.77,398.02,875.46,478.96,950.92,601.15,693.96,790.49,448.30,782.68,518.95,627.78,839.06,760.61,489.76,172.49,353.18,288.55,28.80,45.57,32.55,57.81,32.55,32.81,39.58,36.75,36.75,40.51,39.06,83.91,47.74,167.53,179.11
50%,110159.50,2.46,48.13,51.65,44.23,632.64,75.58,13.64,16.17,15.49,15.08,44.29,45.36,32.52,2.93,420.11,NaN,462.86,462.02,2.53,665.67,399.37,879.70,531.86,981.92,625.87,740.20,861.87,494.47,967.28,564.87,668.98,917.71,878.85,512.27,226.36,473.35,709.67,64.30,49.48,35.42,66.41,34.90,35.16,42.97,40.51,40.22,44.85,42.53,138.02,52.66,193.87,197.34
75%,165239.25,2.50,49.48,52.78,45.31,637.62,80.91,14.54,16.43,15.70,15.12,47.46,49.66,34.94,12.86,421.00,NaN,464.30,466.86,2.59,667.15,400.09,882.13,534.25,1090.81,628.61,750.36,919.10,536.27,1043.98,744.02,697.22,981.25,943.88,555.16,316.84,528.89,837.33,90.82,53.65,39.06,77.86,37.76,36.98,46.61,45.14,44.85,51.22,46.59,208.33,60.76,219.91,216.72
max,220319.00,2.55,56.73,56.03,48.22,800.00,100.00,22.25,23.60,24.35,25.00,76.11,60.00,45.00,31.19,500.00,NaN,739.74,600.00,4.87,878.92,448.91,1107.53,594.06,1227.56,1000.00,839.58,1214.42,2000.00,1841.15,1466.28,1600.00,1800.00,1839.21,1578.60,425.55,694.48,984.06,174.90,417.71,547.92,512.76,420.31,374.22,408.59,1000.00,320.31,370.37,303.53,561.63,464.41,1000.00,1000.00


In [26]:
floats = df.select_dtypes('float64')
df_zscores = (floats.max() - floats.mean()) / floats.std()
df_zscores.sort_values(ascending=False).loc[df_zscores > 3]

sensor_44   82.70
sensor_41   48.74
sensor_38   34.92
sensor_43   33.02
sensor_42   33.02
sensor_39   32.75
sensor_47   24.82
sensor_45   21.59
sensor_49   21.28
sensor_40   20.77
sensor_46   20.61
sensor_50   12.52
sensor_27    8.82
sensor_51    7.28
sensor_33    7.24
sensor_30    5.03
sensor_48    4.99
sensor_09    4.88
sensor_08    4.49
sensor_06    4.04
sensor_32    3.97
sensor_29    3.94
sensor_07    3.52
sensor_13    3.49
sensor_18    3.35
sensor_31    3.30
sensor_28    3.16
sensor_37    3.03
dtype: float64

In [28]:
print(len(df_zscores.loc[df_zscores > 3]))

28


## 5. Column 'machine_status'
Exploration of machine_status shows that the pump can be in three states, NORMAL, BROKEN or RECOVERING. Expected is that during 'broken' the pump is not running. And during 'recovering' all sensors report values outside normal values. For further investigation consider changing the dtype from str to category. 

In [29]:
df['machine_status'].unique()

<StringArray>
['NORMAL', 'BROKEN', 'RECOVERING']
Length: 3, dtype: str